<!--nav--> [🗺 Learning path](README.md) · **15/24** · ◀ [LLM as Judge Evaluation](./LLM_as_Judge_Evaluation.ipynb) · [Simple MultiGPU Multimodal](./Simple_MultiGPU_Multimodal.ipynb) ▶

# Multi-Trace AI Agent Evaluation Engine

A working implementation of a multi-trace agent reasoning + evaluation system across
three benchmark families (ARC-AGI / SWE-Bench / MLE-Bench style mock tasks).

**What this notebook does:**

1. Generates **10 distinct reasoning strategies** per task (greedy, brute-force, heuristic,
   probabilistic, adversarial, counterfactual, constraint-relaxation, minimal-step,
   maximal-exploration, random).
2. Runs each strategy on mock tasks for ARC / SWE / MLE benchmarks.
3. Logs full `state -> action -> observation` traces.
4. Scores every trace on 5 dimensions (correctness / efficiency / robustness /
   generalization / creativity).
5. Visualizes traces as a tree (NetworkX + Matplotlib) and a score heatmap.
6. Picks the **Top-3** trajectories and constructs a **hybrid optimal trace**.
7. Runs **adversarial self-critique** against the best.
8. Performs **beam-search pruning** over partial traces.
9. Exports everything to JSON.
10. Provides optional hooks for real LLMs (Anthropic / OpenAI / HF) — fully runnable
    without any API key.

> **Runs on Colab without modification.** No API keys required for the simulation.


## 0. Setup

In [ ]:
# Colab-safe install (no-op locally if already installed)
%pip install -q networkx matplotlib plotly pandas numpy


In [ ]:
import json, time, random, math, hashlib
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Callable, Tuple
from enum import Enum
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

np.random.seed(0); random.seed(0)
print("Setup complete.")


## 1. Core Data Structures

`Trace` is the central object: a sequence of `Step`s, each capturing
`state -> action -> observation` plus an optional hypothesis the agent held.

In [ ]:
class Benchmark(str, Enum):
    ARC = "arc"
    SWE = "swe"
    MLE = "mle"


@dataclass
class Action:
    tool: str
    args: Dict[str, Any]
    cost: float = 1.0


@dataclass
class Observation:
    result: Any
    error: Optional[str] = None


@dataclass
class Step:
    idx: int
    state_summary: str
    action: Action
    observation: Observation
    hypothesis: Optional[str] = None


@dataclass
class Trace:
    trace_id: str
    benchmark: Benchmark
    strategy: str
    task_id: str
    steps: List[Step] = field(default_factory=list)
    final_answer: Optional[Any] = None
    succeeded: bool = False
    total_cost: float = 0.0
    elapsed_ms: float = 0.0
    scores: Dict[str, float] = field(default_factory=dict)

    def add_step(self, step: Step):
        self.steps.append(step)
        self.total_cost += step.action.cost

    def to_dict(self):
        d = asdict(self)
        d["benchmark"] = self.benchmark.value
        return d


def make_trace(strategy: str, task, benchmark: Benchmark) -> Trace:
    tid = f"{benchmark.value}-{strategy}-{task.task_id}-{random.randint(1000,9999)}"
    return Trace(trace_id=tid, benchmark=benchmark, strategy=strategy, task_id=task.task_id)


## 2. Mock Task Environments

Three deterministic toy environments mirroring the *shape* of the real benchmarks.
They expose `apply()`, `check()`, and a ground-truth `is_correct()` so any strategy
can be evaluated objectively.

In [ ]:
class ARCTask:
    """Discover the grid transformation rule. ARC-AGI style."""
    def __init__(self, task_id="arc-001", true_rule="rotate_90"):
        self.task_id = task_id
        self.input_grid = np.array([[1,0,2],[0,1,0],[2,0,1]])
        self.true_rule = true_rule
        self.candidate_rules = ["identity","rotate_90","rotate_180","rotate_270",
                                "flip_h","flip_v","transpose","color_swap_1_2"]
        self.output_grid = self._apply(self.input_grid, true_rule)

    def _apply(self, g, rule):
        if rule == "identity": return g
        if rule == "rotate_90": return np.rot90(g)
        if rule == "rotate_180": return np.rot90(g, 2)
        if rule == "rotate_270": return np.rot90(g, 3)
        if rule == "flip_h": return np.fliplr(g)
        if rule == "flip_v": return np.flipud(g)
        if rule == "transpose": return g.T
        if rule == "color_swap_1_2":
            r = g.copy(); r[g==1]=2; r[g==2]=1; return r
        return None

    def check(self, rule):
        out = self._apply(self.input_grid, rule)
        return out is not None and np.array_equal(out, self.output_grid)

    def is_correct(self, candidate):
        return candidate == self.true_rule


class SWETask:
    """Localize the buggy file from a bug report. SWE-Bench style (toy)."""
    def __init__(self, task_id="swe-001"):
        self.task_id = task_id
        self.files = ["auth.py","db.py","api.py","utils.py","config.py"]
        self.true_buggy_file = "db.py"
        self.bug_description = "Users get TimeoutError when querying the user table"
        # Ground-truth signal a real grep would surface
        self.file_signals = {"auth.py":["login","token"], "db.py":["timeout","query","conn"],
                             "api.py":["route","endpoint"], "utils.py":["helper","format"],
                             "config.py":["env","setting"]}
        self.relevance = {"auth.py":0.2,"db.py":0.95,"api.py":0.5,"utils.py":0.05,"config.py":0.25}

    def grep(self, keyword):
        kw = keyword.lower()
        return [f for f, sigs in self.file_signals.items() if any(kw in s for s in sigs)]

    def read_file(self, fname):
        if fname == self.true_buggy_file:
            return "def query(): conn.timeout = 1  # FIXME: too short under load"
        return f"# {fname} (no bug here)"

    def is_correct(self, candidate):
        return candidate == self.true_buggy_file


class MLETask:
    """Find the learning rate that minimizes synthetic loss. MLE-Bench style."""
    def __init__(self, task_id="mle-001"):
        self.task_id = task_id
        self.lr_options = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
        self.optimal_lr = 1e-3

    def loss(self, lr):
        # Smooth quadratic in log-space + small noise (deterministic via hash for repeatability)
        rng = np.random.RandomState(int(hashlib.md5(str(lr).encode()).hexdigest()[:8],16) % (2**32))
        return float((np.log10(lr) - np.log10(self.optimal_lr))**2 + 0.02*rng.randn())

    def is_correct(self, candidate):
        return abs(np.log10(candidate) - np.log10(self.optimal_lr)) < 0.5


print("Tasks ready.")


## 3. Ten Reasoning Strategies

Each strategy is a function `(task) -> Trace`. They differ in how they explore the
action space — that's the entire point of multi-trace evaluation.

| Strategy | Behavior |
|---|---|
| greedy | Commit to the most obvious first guess; bail if wrong |
| brute_force | Enumerate everything in the order given |
| heuristic | Use domain knowledge to rank candidates |
| probabilistic | Sample weighted by prior |
| adversarial | Deliberately try the *least* likely candidate first |
| counterfactual | "What if it's NOT X?" — eliminate before commit |
| constraint_relax | Drop a constraint (e.g. ignore color) and check shape only |
| minimal_step | Take the absolute fewest actions possible |
| max_exploration | Compose pairs of actions before committing |
| random | Uniform random sampling |


In [ ]:
# ---------- ARC strategies ----------
def arc_greedy(task):
    t = make_trace("greedy", task, Benchmark.ARC)
    rule = "identity"
    ok = task.check(rule)
    t.add_step(Step(0, "first guess: identity", Action("apply_rule",{"rule":rule},1.0),
                    Observation(ok), "simplest hypothesis"))
    t.final_answer = rule; t.succeeded = task.is_correct(rule)
    return t

def arc_brute_force(task):
    t = make_trace("brute_force", task, Benchmark.ARC)
    for i, rule in enumerate(task.candidate_rules):
        ok = task.check(rule)
        t.add_step(Step(i, f"try {rule}", Action("apply_rule",{"rule":rule},1.0),
                        Observation(ok), f"hypothesis #{i}"))
        if ok:
            t.final_answer = rule; t.succeeded = True; return t
    return t

def arc_heuristic(task):
    t = make_trace("heuristic", task, Benchmark.ARC)
    # Domain prior: rotations are most common in ARC puzzles
    priority = ["rotate_90","rotate_180","flip_h","flip_v","rotate_270","transpose","color_swap_1_2","identity"]
    for i, rule in enumerate(priority):
        ok = task.check(rule)
        t.add_step(Step(i, f"heuristic try {rule}", Action("apply_rule",{"rule":rule},0.9),
                        Observation(ok), "domain prior"))
        if ok:
            t.final_answer = rule; t.succeeded = True; return t
    return t

def arc_probabilistic(task, seed=42):
    rng = random.Random(seed); t = make_trace("probabilistic", task, Benchmark.ARC)
    weights = {"identity":0.05,"rotate_90":0.2,"rotate_180":0.15,"rotate_270":0.1,
               "flip_h":0.15,"flip_v":0.1,"transpose":0.15,"color_swap_1_2":0.1}
    tried = set()
    for i in range(10):
        rules, probs = list(weights.keys()), list(weights.values())
        choice = rng.choices(rules, weights=probs)[0]
        if choice in tried: continue
        tried.add(choice)
        ok = task.check(choice)
        t.add_step(Step(i, f"sample {choice}", Action("apply_rule",{"rule":choice},1.2),
                        Observation(ok), f"p={weights[choice]:.2f}"))
        if ok:
            t.final_answer = choice; t.succeeded = True; return t
    return t

def arc_adversarial(task):
    """Try the LEAST likely candidate first to surface weird cases."""
    t = make_trace("adversarial", task, Benchmark.ARC)
    weird_first = ["color_swap_1_2","identity","transpose","rotate_270",
                   "flip_v","flip_h","rotate_180","rotate_90"]
    for i, rule in enumerate(weird_first):
        ok = task.check(rule)
        t.add_step(Step(i, f"adversarial try {rule}", Action("apply_rule",{"rule":rule},1.1),
                        Observation(ok), "least-likely first"))
        if ok:
            t.final_answer = rule; t.succeeded = True; return t
    return t

def arc_counterfactual(task):
    """Eliminate via 'what if NOT X'. Each elimination is cheaper than a full trial."""
    t = make_trace("counterfactual", task, Benchmark.ARC)
    # Cheap elimination via shape signature first
    in_shape = task.input_grid.shape; out_shape = task.output_grid.shape
    eliminated = set()
    if in_shape == out_shape:
        # rules that change shape can be eliminated cheaply
        for r in []: eliminated.add(r)  # all our rules preserve 3x3
    else:
        for r in ["identity","color_swap_1_2"]: eliminated.add(r)
    t.add_step(Step(0,"shape check",Action("compare_shape",{},0.3),
                    Observation(f"in={in_shape} out={out_shape}"), "shape-based elim"))
    # Now try remaining
    for i, rule in enumerate([r for r in task.candidate_rules if r not in eliminated]):
        ok = task.check(rule)
        t.add_step(Step(i+1, f"after elim try {rule}", Action("apply_rule",{"rule":rule},1.0),
                        Observation(ok), "post-elimination"))
        if ok:
            t.final_answer = rule; t.succeeded = True; return t
    return t

def arc_constraint_relax(task):
    """Match on shape only, ignore exact values, then refine."""
    t = make_trace("constraint_relax", task, Benchmark.ARC)
    # Phase 1: relaxed match (any rule producing same SHAPE counts as candidate)
    candidates = []
    for i, rule in enumerate(task.candidate_rules):
        out = task._apply(task.input_grid, rule)
        same_shape = out is not None and out.shape == task.output_grid.shape
        t.add_step(Step(i, f"relax-check {rule}", Action("shape_match",{"rule":rule},0.4),
                        Observation(same_shape), "shape-relaxed"))
        if same_shape: candidates.append(rule)
    # Phase 2: tight match among candidates
    for j, rule in enumerate(candidates):
        ok = task.check(rule)
        t.add_step(Step(len(t.steps), f"tight {rule}", Action("apply_rule",{"rule":rule},1.0),
                        Observation(ok), "post-relaxation refine"))
        if ok:
            t.final_answer = rule; t.succeeded = True; return t
    return t

def arc_minimal_step(task):
    t = make_trace("minimal_step", task, Benchmark.ARC)
    # Single move: best heuristic guess and stop
    rule = "rotate_90"
    ok = task.check(rule)
    t.add_step(Step(0,"single-shot",Action("apply_rule",{"rule":rule},0.8),
                    Observation(ok), "minimal commit"))
    t.final_answer = rule; t.succeeded = task.is_correct(rule)
    return t

def arc_max_exploration(task):
    """Explore all candidates AND verify by trying compositions."""
    t = make_trace("max_exploration", task, Benchmark.ARC)
    found = None
    for i, rule in enumerate(task.candidate_rules):
        ok = task.check(rule)
        t.add_step(Step(i, f"explore {rule}", Action("apply_rule",{"rule":rule},1.0),
                        Observation(ok), "exhaustive"))
        if ok and found is None: found = rule
    # Always also probe a few compositions for confidence
    for j, (a,b) in enumerate([("rotate_90","rotate_90"),("flip_h","flip_v")]):
        out = task._apply(task._apply(task.input_grid,a), b)
        ok = out is not None and np.array_equal(out, task.output_grid)
        t.add_step(Step(len(t.steps), f"compose {a}+{b}", Action("compose",{"a":a,"b":b},1.5),
                        Observation(ok), "composition probe"))
    t.final_answer = found; t.succeeded = found is not None and task.is_correct(found)
    return t

def arc_random(task, seed=7):
    rng = random.Random(seed); t = make_trace("random", task, Benchmark.ARC)
    pool = list(task.candidate_rules); rng.shuffle(pool)
    for i, rule in enumerate(pool):
        ok = task.check(rule)
        t.add_step(Step(i, f"random {rule}", Action("apply_rule",{"rule":rule},1.0),
                        Observation(ok), "uniform sample"))
        if ok:
            t.final_answer = rule; t.succeeded = True; return t
    return t


ARC_STRATEGIES = {
    "greedy": arc_greedy, "brute_force": arc_brute_force, "heuristic": arc_heuristic,
    "probabilistic": arc_probabilistic, "adversarial": arc_adversarial,
    "counterfactual": arc_counterfactual, "constraint_relax": arc_constraint_relax,
    "minimal_step": arc_minimal_step, "max_exploration": arc_max_exploration, "random": arc_random,
}
print(f"Registered {len(ARC_STRATEGIES)} ARC strategies.")


In [ ]:
# ---------- SWE strategies (bug localization) ----------
def swe_greedy(task):
    t = make_trace("greedy", task, Benchmark.SWE)
    guess = task.files[0]
    t.add_step(Step(0,"first file",Action("inspect",{"file":guess},1.0),
                    Observation(task.read_file(guess)), "alphabetical"))
    t.final_answer = guess; t.succeeded = task.is_correct(guess); return t

def swe_brute_force(task):
    t = make_trace("brute_force", task, Benchmark.SWE)
    for i,f in enumerate(task.files):
        content = task.read_file(f); has_fixme = "FIXME" in content
        t.add_step(Step(i,f"read {f}",Action("read_file",{"file":f},1.0),
                        Observation({"snippet":content[:60],"has_fixme":has_fixme})))
        if has_fixme:
            t.final_answer = f; t.succeeded = task.is_correct(f); return t
    return t

def swe_heuristic(task):
    """Grep for keyword from bug description first."""
    t = make_trace("heuristic", task, Benchmark.SWE)
    keyword = "timeout"
    matches = task.grep(keyword)
    t.add_step(Step(0,f"grep '{keyword}'",Action("grep",{"kw":keyword},0.5),
                    Observation(matches), "extracted from bug report"))
    for i,f in enumerate(matches):
        content = task.read_file(f)
        t.add_step(Step(i+1,f"read {f}",Action("read_file",{"file":f},1.0),
                        Observation(content[:60])))
        if "FIXME" in content:
            t.final_answer = f; t.succeeded = task.is_correct(f); return t
    return t

def swe_probabilistic(task, seed=3):
    rng = random.Random(seed); t = make_trace("probabilistic", task, Benchmark.SWE)
    weights = list(task.relevance.values()); files = list(task.relevance.keys())
    tried = set()
    for i in range(8):
        f = rng.choices(files, weights=weights)[0]
        if f in tried: continue
        tried.add(f); content = task.read_file(f)
        t.add_step(Step(i,f"sample {f}",Action("read_file",{"file":f},1.2),
                        Observation(content[:60]), f"p={task.relevance[f]:.2f}"))
        if "FIXME" in content:
            t.final_answer = f; t.succeeded = task.is_correct(f); return t
    return t

def swe_adversarial(task):
    """Inspect lowest-relevance files first to challenge prior."""
    t = make_trace("adversarial", task, Benchmark.SWE)
    order = sorted(task.files, key=lambda f: task.relevance[f])
    for i,f in enumerate(order):
        content = task.read_file(f)
        t.add_step(Step(i,f"adversarial {f}",Action("read_file",{"file":f},1.1),
                        Observation(content[:60]), "low-prior first"))
        if "FIXME" in content:
            t.final_answer = f; t.succeeded = task.is_correct(f); return t
    return t

def swe_counterfactual(task):
    t = make_trace("counterfactual", task, Benchmark.SWE)
    # Eliminate files with no relevant signal first
    irrelevant = [f for f,sigs in task.file_signals.items() if not any("timeout" in s or "query" in s for s in sigs)]
    t.add_step(Step(0,"signal scan",Action("scan_signals",{},0.4),
                    Observation({"eliminated":irrelevant}), "what if NOT in irrelevant?"))
    remaining = [f for f in task.files if f not in irrelevant]
    for i,f in enumerate(remaining):
        content = task.read_file(f)
        t.add_step(Step(i+1,f"check {f}",Action("read_file",{"file":f},1.0),
                        Observation(content[:60])))
        if "FIXME" in content:
            t.final_answer = f; t.succeeded = task.is_correct(f); return t
    return t

def swe_constraint_relax(task):
    """Drop the 'must contain FIXME' constraint, check just keyword presence."""
    t = make_trace("constraint_relax", task, Benchmark.SWE)
    candidates = []
    for i,f in enumerate(task.files):
        c = task.read_file(f); has_kw = "timeout" in c.lower()
        t.add_step(Step(i,f"relaxed-check {f}",Action("read_file",{"file":f},0.6),
                        Observation({"has_timeout":has_kw}), "relaxed predicate"))
        if has_kw: candidates.append(f)
    for j,f in enumerate(candidates):
        c = task.read_file(f)
        t.add_step(Step(len(t.steps),f"tight {f}",Action("read_file",{"file":f},1.0),
                        Observation(c[:60]), "tight predicate"))
        if "FIXME" in c:
            t.final_answer = f; t.succeeded = task.is_correct(f); return t
    return t

def swe_minimal_step(task):
    t = make_trace("minimal_step", task, Benchmark.SWE)
    # Best a-priori guess, check, done
    f = max(task.relevance, key=task.relevance.get)
    c = task.read_file(f)
    t.add_step(Step(0,f"prior-best {f}",Action("read_file",{"file":f},0.8),
                    Observation(c[:60]), "max-prior"))
    t.final_answer = f; t.succeeded = task.is_correct(f); return t

def swe_max_exploration(task):
    t = make_trace("max_exploration", task, Benchmark.SWE)
    # Read every file AND grep multiple keywords
    found = None
    for kw in ["timeout","query","conn","login"]:
        matches = task.grep(kw)
        t.add_step(Step(len(t.steps),f"grep {kw}",Action("grep",{"kw":kw},0.5),
                        Observation(matches)))
    for i,f in enumerate(task.files):
        c = task.read_file(f)
        t.add_step(Step(len(t.steps),f"read {f}",Action("read_file",{"file":f},1.0),
                        Observation(c[:60])))
        if "FIXME" in c and found is None: found = f
    t.final_answer = found; t.succeeded = found is not None and task.is_correct(found); return t

def swe_random(task, seed=11):
    rng = random.Random(seed); t = make_trace("random", task, Benchmark.SWE)
    order = list(task.files); rng.shuffle(order)
    for i,f in enumerate(order):
        c = task.read_file(f)
        t.add_step(Step(i,f"random {f}",Action("read_file",{"file":f},1.0),
                        Observation(c[:60])))
        if "FIXME" in c:
            t.final_answer = f; t.succeeded = task.is_correct(f); return t
    return t


SWE_STRATEGIES = {
    "greedy": swe_greedy, "brute_force": swe_brute_force, "heuristic": swe_heuristic,
    "probabilistic": swe_probabilistic, "adversarial": swe_adversarial,
    "counterfactual": swe_counterfactual, "constraint_relax": swe_constraint_relax,
    "minimal_step": swe_minimal_step, "max_exploration": swe_max_exploration, "random": swe_random,
}
print(f"Registered {len(SWE_STRATEGIES)} SWE strategies.")


In [ ]:
# ---------- MLE strategies (hyperparameter search) ----------
def mle_greedy(task):
    t = make_trace("greedy", task, Benchmark.MLE)
    lr = 1e-2  # naive first guess
    loss = task.loss(lr)
    t.add_step(Step(0,f"try lr={lr}",Action("train",{"lr":lr},2.0),Observation(loss),"naive guess"))
    t.final_answer = lr; t.succeeded = task.is_correct(lr); return t

def mle_brute_force(task):
    t = make_trace("brute_force", task, Benchmark.MLE)
    best_lr, best_loss = None, float("inf")
    for i,lr in enumerate(task.lr_options):
        loss = task.loss(lr)
        t.add_step(Step(i,f"train lr={lr}",Action("train",{"lr":lr},2.0),Observation(loss)))
        if loss < best_loss: best_loss, best_lr = loss, lr
    t.final_answer = best_lr; t.succeeded = task.is_correct(best_lr); return t

def mle_heuristic(task):
    """Try canonical 1e-3 first (literature prior)."""
    t = make_trace("heuristic", task, Benchmark.MLE)
    order = [1e-3, 1e-4, 1e-2, 1e-5, 1e-1]; best_lr,best_loss = None,float("inf")
    for i,lr in enumerate(order):
        loss = task.loss(lr)
        t.add_step(Step(i,f"prior-order {lr}",Action("train",{"lr":lr},1.8),Observation(loss),"lit prior"))
        if loss < best_loss: best_loss, best_lr = loss, lr
        if loss < 0.1: break  # early-stop on confidence
    t.final_answer = best_lr; t.succeeded = task.is_correct(best_lr); return t

def mle_probabilistic(task, seed=5):
    rng = random.Random(seed); t = make_trace("probabilistic", task, Benchmark.MLE)
    best_lr,best_loss = None,float("inf")
    for i in range(5):
        lr = rng.choice(task.lr_options); loss = task.loss(lr)
        t.add_step(Step(i,f"sample {lr}",Action("train",{"lr":lr},2.2),Observation(loss),"random sample"))
        if loss < best_loss: best_loss, best_lr = loss, lr
    t.final_answer = best_lr; t.succeeded = task.is_correct(best_lr); return t

def mle_adversarial(task):
    """Try the worst-looking lrs first to learn the loss surface."""
    t = make_trace("adversarial", task, Benchmark.MLE)
    order = [1e-1, 1e-5, 1e-2, 1e-4, 1e-3]; best_lr,best_loss = None,float("inf")
    for i,lr in enumerate(order):
        loss = task.loss(lr)
        t.add_step(Step(i,f"adversarial {lr}",Action("train",{"lr":lr},2.1),Observation(loss),"extremes first"))
        if loss < best_loss: best_loss, best_lr = loss, lr
    t.final_answer = best_lr; t.succeeded = task.is_correct(best_lr); return t

def mle_counterfactual(task):
    """Probe extremes to BOUND the optimum, then bisect."""
    t = make_trace("counterfactual", task, Benchmark.MLE)
    lo, hi = 1e-5, 1e-1
    l_lo, l_hi = task.loss(lo), task.loss(hi)
    t.add_step(Step(0,f"bound lo={lo}",Action("train",{"lr":lo},2.0),Observation(l_lo),"establish bound"))
    t.add_step(Step(1,f"bound hi={hi}",Action("train",{"lr":hi},2.0),Observation(l_hi),"establish bound"))
    # bisect in log-space
    candidates = [1e-4,1e-3,1e-2]; best_lr,best_loss=None,float("inf")
    for i,lr in enumerate(candidates):
        loss = task.loss(lr)
        t.add_step(Step(i+2,f"bisect {lr}",Action("train",{"lr":lr},2.0),Observation(loss),"between bounds"))
        if loss < best_loss: best_loss, best_lr = loss, lr
    t.final_answer = best_lr; t.succeeded = task.is_correct(best_lr); return t

def mle_constraint_relax(task):
    """Run cheap 1-epoch proxy first, then full train on top-2."""
    t = make_trace("constraint_relax", task, Benchmark.MLE)
    proxy_losses = {lr: task.loss(lr) + 0.2 for lr in task.lr_options}  # noisier proxy
    for i,(lr,pl) in enumerate(proxy_losses.items()):
        t.add_step(Step(i,f"proxy lr={lr}",Action("proxy_train",{"lr":lr},0.5),
                        Observation(pl), "cheap proxy"))
    top2 = sorted(proxy_losses, key=proxy_losses.get)[:2]
    best_lr,best_loss=None,float("inf")
    for j,lr in enumerate(top2):
        loss = task.loss(lr)
        t.add_step(Step(len(t.steps),f"full {lr}",Action("train",{"lr":lr},2.0),
                        Observation(loss), "full after relaxed"))
        if loss < best_loss: best_loss, best_lr = loss, lr
    t.final_answer = best_lr; t.succeeded = task.is_correct(best_lr); return t

def mle_minimal_step(task):
    t = make_trace("minimal_step", task, Benchmark.MLE)
    lr = 1e-3
    loss = task.loss(lr)
    t.add_step(Step(0,f"single {lr}",Action("train",{"lr":lr},1.5),Observation(loss),"prior-best only"))
    t.final_answer = lr; t.succeeded = task.is_correct(lr); return t

def mle_max_exploration(task):
    """Sweep + repeated trials for variance estimation."""
    t = make_trace("max_exploration", task, Benchmark.MLE)
    runs = defaultdict(list)
    for trial in range(2):
        for i,lr in enumerate(task.lr_options):
            loss = task.loss(lr); runs[lr].append(loss)
            t.add_step(Step(len(t.steps),f"trial{trial} lr={lr}",
                            Action("train",{"lr":lr,"trial":trial},2.0),
                            Observation(loss), "variance probe"))
    avg = {lr: np.mean(v) for lr,v in runs.items()}
    best_lr = min(avg, key=avg.get)
    t.final_answer = best_lr; t.succeeded = task.is_correct(best_lr); return t

def mle_random(task, seed=13):
    rng = random.Random(seed); t = make_trace("random", task, Benchmark.MLE)
    order = list(task.lr_options); rng.shuffle(order)
    best_lr,best_loss=None,float("inf")
    for i,lr in enumerate(order[:4]):
        loss = task.loss(lr)
        t.add_step(Step(i,f"random {lr}",Action("train",{"lr":lr},2.0),Observation(loss)))
        if loss < best_loss: best_loss, best_lr = loss, lr
    t.final_answer = best_lr; t.succeeded = task.is_correct(best_lr); return t


MLE_STRATEGIES = {
    "greedy": mle_greedy, "brute_force": mle_brute_force, "heuristic": mle_heuristic,
    "probabilistic": mle_probabilistic, "adversarial": mle_adversarial,
    "counterfactual": mle_counterfactual, "constraint_relax": mle_constraint_relax,
    "minimal_step": mle_minimal_step, "max_exploration": mle_max_exploration, "random": mle_random,
}
print(f"Registered {len(MLE_STRATEGIES)} MLE strategies.")


## 4. Scoring System

Five dimensions, each `0..10`. `total` is the unweighted mean — change `WEIGHTS`
to bias towards correctness, efficiency, etc.

In [ ]:
WEIGHTS = {
    "correctness":   0.40,
    "efficiency":    0.20,
    "robustness":    0.15,
    "generalization":0.10,
    "creativity":    0.15,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9


def score_trace(trace: Trace) -> Dict[str, float]:
    n_steps = max(1, len(trace.steps))
    correctness = 10.0 if trace.succeeded else 0.0
    efficiency = max(0.0, 10.0 - trace.total_cost)
    errors = sum(1 for s in trace.steps if s.observation.error)
    robustness = (10.0 - errors*2) if trace.succeeded else max(0.0, 5.0 - errors)
    distinct_tools = len({s.action.tool for s in trace.steps})
    generalization = min(10.0, distinct_tools * 3.0)
    # Action-type entropy as creativity proxy
    counts = Counter(s.action.tool for s in trace.steps)
    probs = np.array(list(counts.values())) / n_steps
    entropy = float(-np.sum(probs * np.log2(probs + 1e-12)))
    creativity = min(10.0, entropy * 4.0 + (3.0 if trace.succeeded else 0.0))

    scores = {"correctness":correctness, "efficiency":efficiency,
              "robustness":robustness, "generalization":generalization,
              "creativity":creativity}
    weighted_total = sum(scores[k] * WEIGHTS[k] for k in WEIGHTS)
    scores["weighted_total"] = round(weighted_total, 3)
    trace.scores = {k: round(v,3) for k,v in scores.items()}
    return trace.scores


print("Scoring ready.")


## 5. Run the Multi-Trace Experiment

Generate every strategy on every benchmark and score them.

In [ ]:
def run_all() -> List[Trace]:
    traces: List[Trace] = []
    suites = [
        (ARCTask(), ARC_STRATEGIES),
        (SWETask(), SWE_STRATEGIES),
        (MLETask(), MLE_STRATEGIES),
    ]
    for task, strategies in suites:
        for name, fn in strategies.items():
            t0 = time.perf_counter()
            tr = fn(task)
            tr.elapsed_ms = (time.perf_counter() - t0) * 1000
            score_trace(tr)
            traces.append(tr)
    return traces


traces = run_all()

# Build a results dataframe
rows = []
for t in traces:
    rows.append({
        "benchmark": t.benchmark.value, "strategy": t.strategy,
        "succeeded": t.succeeded, "steps": len(t.steps),
        "cost": round(t.total_cost,2), "ms": round(t.elapsed_ms,2),
        **t.scores,
    })
df = pd.DataFrame(rows).sort_values(["benchmark","weighted_total"], ascending=[True,False])
df.head(40)


## 6. Visualize: Score Heatmap + Trajectory Tree

In [ ]:
# --- Heatmap: weighted_total per (benchmark, strategy) ---
pivot = df.pivot(index="strategy", columns="benchmark", values="weighted_total")
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i,j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                    color="white" if v < 5 else "black", fontsize=9)
plt.colorbar(im, ax=ax, label="weighted total")
ax.set_title("Strategy x Benchmark — weighted total score")
plt.tight_layout(); plt.show()


In [ ]:
# --- Trajectory tree: ARC strategies as branches from a shared root ---
G = nx.DiGraph()
arc_traces = [t for t in traces if t.benchmark == Benchmark.ARC]
G.add_node("ARC-task", label="ARC root")

for tr in arc_traces:
    prev = "ARC-task"
    for s in tr.steps:
        node = f"{tr.strategy}#{s.idx}"
        ok = "OK" if s.observation.result is True else ("MATCH" if s.observation.result else "miss")
        G.add_node(node, label=f"{tr.strategy}\n{s.action.tool}\n{ok}")
        G.add_edge(prev, node)
        prev = node
    final = f"{tr.strategy}::final"
    G.add_node(final, label=f"{tr.strategy}\nfinal={tr.final_answer}\n{'YES' if tr.succeeded else 'NO'}")
    G.add_edge(prev, final)

plt.figure(figsize=(18, 10))
try:
    pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
except Exception:
    pos = nx.spring_layout(G, k=1.2, iterations=80, seed=7)
node_colors = []
for n in G.nodes():
    d = G.nodes[n].get("label","")
    if "YES" in d: node_colors.append("#4CAF50")
    elif "NO" in d: node_colors.append("#E53935")
    elif n == "ARC-task": node_colors.append("#1E88E5")
    else: node_colors.append("#FFB300")
nx.draw(G, pos, with_labels=False, node_color=node_colors, node_size=320,
        edge_color="#888", arrows=True, arrowsize=8)
labels = {n: G.nodes[n].get("label","") for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=6)
plt.title("ARC strategies — full reasoning trees (green=success, red=failure)")
plt.axis("off"); plt.tight_layout(); plt.show()


In [ ]:
# --- Plotly grouped bar (interactive) ---
if HAS_PLOTLY:
    try:
        fig = go.Figure()
        for bm in df["benchmark"].unique():
            sub = df[df["benchmark"] == bm].reset_index(drop=True)
            fig.add_trace(go.Bar(name=bm, x=sub["strategy"].tolist(),
                                 y=sub["weighted_total"].tolist(),
                                 hovertext=[f"steps={s} cost={c} ok={ok}"
                                            for s,c,ok in zip(sub["steps"], sub["cost"], sub["succeeded"])]))
        fig.update_layout(barmode="group", height=480,
                          title="Weighted total score by strategy & benchmark",
                          xaxis_title="strategy", yaxis_title="weighted total")
        fig.show()
    except Exception as e:
        print(f"plotly chart failed ({e}); matplotlib heatmap above already covers this view.")
else:
    print("plotly not available; skipping interactive chart.")


## 7. Optimal Path Selection + Hybrid Construction

Rank within each benchmark, take the Top-3, then synthesize a `HybridTrace` that
keeps the strongest *prefix* of the winner and grafts diverse exploration steps
from the runners-up. The hybrid is scored alongside the originals.

In [ ]:
def top_k(traces, benchmark, k=3):
    pool = [t for t in traces if t.benchmark == benchmark]
    return sorted(pool, key=lambda t: t.scores["weighted_total"], reverse=True)[:k]


def build_hybrid(top: List[Trace]) -> Trace:
    """Take the winner's successful action chain, then add diverse probes from runners-up."""
    winner = top[0]
    hybrid = Trace(
        trace_id=f"hybrid-{winner.benchmark.value}-{random.randint(1000,9999)}",
        benchmark=winner.benchmark, strategy="HYBRID",
        task_id=winner.task_id,
    )
    # 1. copy the winning prefix
    for s in winner.steps:
        hybrid.add_step(Step(len(hybrid.steps), s.state_summary, s.action,
                             s.observation, f"[from {winner.strategy}] {s.hypothesis or ''}"))
    # 2. graft exploration probes from runners-up that introduce new tools
    seen_tools = {s.action.tool for s in hybrid.steps}
    for runner in top[1:]:
        for s in runner.steps:
            if s.action.tool not in seen_tools:
                hybrid.add_step(Step(len(hybrid.steps), f"graft from {runner.strategy}",
                                     s.action, s.observation,
                                     f"diversity from {runner.strategy}"))
                seen_tools.add(s.action.tool)
    hybrid.final_answer = winner.final_answer
    hybrid.succeeded = winner.succeeded
    score_trace(hybrid)
    return hybrid


print("\n=== TOP 3 + HYBRID per benchmark ===\n")
hybrids = {}
for bm in Benchmark:
    top = top_k(traces, bm, 3)
    hyb = build_hybrid(top)
    hybrids[bm] = hyb
    print(f"--- {bm.value.upper()} ---")
    for rank, t in enumerate(top, 1):
        print(f"  #{rank} {t.strategy:18s}  total={t.scores['weighted_total']:.2f}  "
              f"steps={len(t.steps):2d}  cost={t.total_cost:.1f}  ok={t.succeeded}")
    print(f"  HYBRID            total={hyb.scores['weighted_total']:.2f}  "
          f"steps={len(hyb.steps):2d}  cost={hyb.total_cost:.1f}  ok={hyb.succeeded}\n")


## 8. Adversarial Self-Critique

For each top trace, surface failure modes and unjustified assumptions. Where the
hybrid loses to a single-strategy trace, we record the critique and emit a revision
suggestion.

In [ ]:
def adversarial_critique(trace: Trace) -> Dict[str, Any]:
    weaknesses = []
    if not trace.succeeded:
        weaknesses.append("Did not converge — final answer is wrong.")
    if trace.total_cost > 6:
        weaknesses.append(f"High cost ({trace.total_cost:.1f}) — wastes budget.")
    if len({s.action.tool for s in trace.steps}) <= 1:
        weaknesses.append("Single-tool strategy — fragile to environment changes.")
    if len(trace.steps) > 12:
        weaknesses.append("Long trajectory — likely over-exploration.")
    early_match = next((i for i,s in enumerate(trace.steps)
                        if s.observation.result is True), None)
    if early_match is not None and early_match < len(trace.steps) - 1:
        weaknesses.append(f"Found match at step {early_match} but kept exploring.")
    if not weaknesses:
        weaknesses.append("No obvious failure mode under current scoring.")
    return {
        "trace_id": trace.trace_id, "strategy": trace.strategy,
        "succeeded": trace.succeeded, "weaknesses": weaknesses,
        "suggested_revision": "Early-stop on first success; cap budget; diversify tool palette.",
    }


print("=== ADVERSARIAL CRITIQUES ===")
for bm, hyb in hybrids.items():
    crit = adversarial_critique(hyb)
    print(f"\n[{bm.value.upper()}] HYBRID critique:")
    for w in crit["weaknesses"]:
        print(f"  - {w}")
    print(f"  -> {crit['suggested_revision']}")


## 9. Beam-Search Pruning Over Partial Traces

Frontier-style search: at each depth, keep the top-`beam_width` partial traces by
running score, expand each by one action, repeat. This is the "tree of thoughts"
shape applied to executable traces.

In [ ]:
def beam_search_arc(task, beam_width=3, max_depth=4):
    """Beam over ARC rule trials. Each beam state = ordered set of tried rules."""
    initial = (tuple(), 0.0)  # (tried, partial_score)
    beam = [initial]
    log = []

    for depth in range(max_depth):
        candidates = []
        for tried, score in beam:
            remaining = [r for r in task.candidate_rules if r not in tried]
            for rule in remaining:
                ok = task.check(rule)
                # partial score: progress + correctness signal - cost
                step_score = score + (5.0 if ok else 0.4) - 0.3
                candidates.append((tried + (rule,), step_score, rule, ok))
        # keep top beam_width
        candidates.sort(key=lambda x: x[1], reverse=True)
        beam = [(c[0], c[1]) for c in candidates[:beam_width]]
        log.append({"depth": depth, "kept": [{"path": list(t),
                                              "score": round(s,2)} for t,s in beam]})
        # early stop if any beam state contains the true rule
        if any(any(task.is_correct(r) for r in t) for t,_ in beam):
            break
    return beam, log


beam, beam_log = beam_search_arc(ARCTask(), beam_width=3, max_depth=4)
print("=== BEAM SEARCH (ARC, width=3) ===")
for entry in beam_log:
    print(f"depth {entry['depth']}:")
    for k in entry["kept"]:
        print(f"  score={k['score']:5.2f}  path={k['path']}")


## 10. Export Everything to JSON

Single artifact you can post-process, diff between runs, or feed to a downstream
analytics layer.

In [ ]:
out = {
    "config": {"weights": WEIGHTS, "n_strategies": len(ARC_STRATEGIES)},
    "traces": [t.to_dict() for t in traces],
    "summary_table": df.to_dict(orient="records"),
    "top_per_benchmark": {bm.value: [t.trace_id for t in top_k(traces, bm, 3)]
                          for bm in Benchmark},
    "hybrids": {bm.value: hybrids[bm].to_dict() for bm in Benchmark},
    "critiques": {bm.value: adversarial_critique(hybrids[bm]) for bm in Benchmark},
    "beam_search_log": beam_log,
}

# Action objects need conversion
def _default(o):
    if hasattr(o, "to_dict"): return o.to_dict()
    if hasattr(o, "__dict__"): return o.__dict__
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.int64, np.int32)): return int(o)
    if isinstance(o, (np.float64, np.float32)): return float(o)
    return str(o)

with open("agent_trace_evaluation.json", "w") as f:
    json.dump(out, f, indent=2, default=_default)

import os
print(f"Wrote agent_trace_evaluation.json ({os.path.getsize('agent_trace_evaluation.json')} bytes)")


## 11. (Optional) Real LLM Hooks

Drop-in adapters for Anthropic / OpenAI / Hugging Face. Leave them un-keyed and the
notebook still runs end-to-end on the mock environment above; set the env var to
swap in a real model for trace generation.

In [ ]:
import os

def llm_anthropic(prompt: str, model: str = "claude-opus-4-7") -> str:
    """Anthropic SDK call. Requires ANTHROPIC_API_KEY in env."""
    if not os.environ.get("ANTHROPIC_API_KEY"):
        return "[anthropic: no ANTHROPIC_API_KEY set — skipping]"
    try:
        import anthropic
    except ImportError:
        return "[anthropic SDK not installed: pip install anthropic]"
    client = anthropic.Anthropic()
    msg = client.messages.create(model=model, max_tokens=512,
                                 messages=[{"role":"user","content":prompt}])
    return msg.content[0].text


def llm_openai(prompt: str, model: str = "gpt-4o-mini") -> str:
    if not os.environ.get("OPENAI_API_KEY"):
        return "[openai: no OPENAI_API_KEY set — skipping]"
    try:
        from openai import OpenAI
    except ImportError:
        return "[openai SDK not installed: pip install openai]"
    client = OpenAI()
    r = client.chat.completions.create(model=model,
                                        messages=[{"role":"user","content":prompt}])
    return r.choices[0].message.content


def llm_hf(prompt: str, model: str = "mistralai/Mistral-7B-Instruct-v0.2") -> str:
    if not os.environ.get("HF_TOKEN"):
        return "[hf: no HF_TOKEN set — skipping]"
    try:
        from huggingface_hub import InferenceClient
    except ImportError:
        return "[huggingface_hub not installed: pip install huggingface_hub]"
    client = InferenceClient(token=os.environ["HF_TOKEN"])
    return client.text_generation(prompt, model=model, max_new_tokens=400)


# Example: ask any of them to critique the hybrid trace verbally.
sample_prompt = (
    "You are an AI agent reasoning critic. Given this trajectory summary, list 3 weaknesses "
    "and propose a stronger strategy in <100 words.\n\n"
    f"Trajectory: {[s.action.tool for s in hybrids[Benchmark.SWE].steps]}\n"
    f"Succeeded: {hybrids[Benchmark.SWE].succeeded}\n"
)
print("--- llm_anthropic ---"); print(llm_anthropic(sample_prompt))
print("\n--- llm_openai ---"); print(llm_openai(sample_prompt))
print("\n--- llm_hf ---"); print(llm_hf(sample_prompt))


## 12. Agent Trace Evaluator + Concrete Retry Suggestions

This is the **closed-loop piece**. Given any (possibly failed) trace, the
`TraceEvaluator`:

1. **Diagnoses** failure modes — concrete categories with evidence and severity:
   `wrong_final`, `premature_commit`, `no_diversity`, `missed_signal`,
   `over_exploration`, `incomplete_exploration`, `none`.
2. **Suggests retry actions** as actual `Action` objects (not vague advice) —
   each is a tool call with args the agent can execute directly, ranked by
   expected gain.
3. **Executes** those retries against the task and **re-scores**.
4. Repeats up to `max_retries` until success or until no useful actions remain.

The output is a chain `[initial -> retry1 -> retry2 -> ...]` where each link is a
real `Trace` you can compare side-by-side.

In [ ]:
@dataclass
class FailureMode:
    category: str
    evidence: str
    severity: float = 0.5


@dataclass
class RetryAction:
    description: str
    concrete_action: Action
    rationale: str
    expected_gain: float = 0.5


@dataclass
class RetryPlan:
    trace_id: str
    diagnosis: List[FailureMode] = field(default_factory=list)
    actions: List[RetryAction] = field(default_factory=list)
    summary: str = ""


class TraceEvaluator:
    """Diagnose -> suggest concrete retry actions -> execute -> re-score."""

    # ----- 1. Diagnose -----
    def diagnose(self, trace: Trace, task) -> List[FailureMode]:
        modes: List[FailureMode] = []
        if not trace.succeeded:
            modes.append(FailureMode("wrong_final",
                f"final={trace.final_answer!r} != ground truth", 1.0))
        if len(trace.steps) == 1 and not trace.succeeded:
            modes.append(FailureMode("premature_commit",
                "single-step trace, no exploration", 0.8))
        tools = {s.action.tool for s in trace.steps}
        if len(tools) == 1 and len(trace.steps) > 2 and not trace.succeeded:
            modes.append(FailureMode("no_diversity",
                f"only tool used: {next(iter(tools))}", 0.6))
        for i, s in enumerate(trace.steps):
            if s.observation.result is True and i < len(trace.steps) - 1:
                if not trace.succeeded:
                    modes.append(FailureMode("missed_signal",
                        f"step {i} returned True but trace didn't commit", 0.9))
                break
        if trace.succeeded and trace.total_cost > 8 and len(trace.steps) > 8:
            modes.append(FailureMode("over_exploration",
                f"{len(trace.steps)} steps, cost={trace.total_cost:.1f}", 0.3))
        if isinstance(task, ARCTask):
            tried = {s.action.args.get("rule") for s in trace.steps}
            tried.discard(None)
            untried = [r for r in task.candidate_rules if r not in tried]
            if untried and not trace.succeeded:
                modes.append(FailureMode("incomplete_exploration",
                    f"untried rules: {untried}", 0.7))
        if not modes:
            modes.append(FailureMode("none", "no failure detected", 0.0))
        return modes

    # ----- 2. Suggest concrete actions -----
    def suggest(self, trace: Trace, task) -> RetryPlan:
        diag = self.diagnose(trace, task)
        actions: List[RetryAction] = []

        if isinstance(task, ARCTask):
            tried = {s.action.args.get("rule") for s in trace.steps}
            untried = [r for r in task.candidate_rules if r not in tried]
            # Heuristic prior: rotations more likely
            prior = {"rotate_90":0.9,"rotate_180":0.7,"rotate_270":0.6,
                     "flip_h":0.6,"flip_v":0.5,"transpose":0.5,
                     "color_swap_1_2":0.3,"identity":0.1}
            for rule in untried:
                actions.append(RetryAction(
                    description=f"Try untried rule: {rule}",
                    concrete_action=Action("apply_rule", {"rule": rule}, 1.0),
                    rationale=f"unattempted; prior={prior.get(rule,0.5):.2f}",
                    expected_gain=prior.get(rule, 0.5),
                ))
            # Composition probes when single rules exhausted
            if not untried:
                actions.append(RetryAction(
                    description="Compose rotate_90 + rotate_90",
                    concrete_action=Action("compose",
                        {"a":"rotate_90","b":"rotate_90"}, 1.5),
                    rationale="single rules exhausted; expand search to compositions",
                    expected_gain=0.4,
                ))

        elif isinstance(task, SWETask):
            inspected = {s.action.args.get("file")
                         for s in trace.steps if s.action.tool == "read_file"}
            uninspected = [f for f in task.files if f not in inspected]
            for f in uninspected:
                rel = task.relevance.get(f, 0)
                actions.append(RetryAction(
                    description=f"Read uninspected file: {f}",
                    concrete_action=Action("read_file", {"file": f}, 1.0),
                    rationale=f"prior relevance={rel:.2f}",
                    expected_gain=rel,
                ))
            kws_used = {s.action.args.get("kw")
                        for s in trace.steps if s.action.tool == "grep"}
            for kw in ["timeout","query","conn"]:
                if kw not in kws_used:
                    actions.append(RetryAction(
                        description=f"Grep for keyword: '{kw}'",
                        concrete_action=Action("grep", {"kw": kw}, 0.5),
                        rationale="bug-description keyword not yet searched",
                        expected_gain=0.7 if kw == "timeout" else 0.5,
                    ))

        elif isinstance(task, MLETask):
            tried_lrs = {s.action.args.get("lr")
                         for s in trace.steps if s.action.tool == "train"}
            untried = [lr for lr in task.lr_options if lr not in tried_lrs]
            for lr in untried:
                # higher gain for lrs near the typical optimum 1e-3
                gain = 1.0 / (1.0 + abs(np.log10(lr) - np.log10(1e-3)))
                actions.append(RetryAction(
                    description=f"Train with lr={lr}",
                    concrete_action=Action("train", {"lr": lr}, 2.0),
                    rationale=f"untried lr; centred-prior gain={gain:.2f}",
                    expected_gain=gain,
                ))

        actions.sort(key=lambda a: a.expected_gain, reverse=True)
        actions = actions[:8]  # cap

        summary = (f"diagnosed {len(diag)} mode(s) "
                   f"({', '.join(d.category for d in diag)}); "
                   f"proposing {len(actions)} concrete retry action(s)")
        return RetryPlan(trace.trace_id, diag, actions, summary)

    # ----- 3. Execute -----
    def _execute(self, action: Action, task) -> Observation:
        try:
            if action.tool == "apply_rule":
                return Observation(task.check(action.args["rule"]))
            if action.tool == "compose" and isinstance(task, ARCTask):
                out = task._apply(task._apply(task.input_grid, action.args["a"]),
                                  action.args["b"])
                return Observation(out is not None and
                                   np.array_equal(out, task.output_grid))
            if action.tool == "read_file" and isinstance(task, SWETask):
                return Observation(task.read_file(action.args["file"]))
            if action.tool == "grep" and isinstance(task, SWETask):
                return Observation(task.grep(action.args["kw"]))
            if action.tool == "train" and isinstance(task, MLETask):
                return Observation(task.loss(action.args["lr"]))
        except Exception as e:
            return Observation(None, error=str(e))
        return Observation(None, error=f"unknown tool: {action.tool}")

    def _extract_answer(self, action: Action, obs: Observation, task):
        if action.tool == "apply_rule" and obs.result is True:
            return action.args["rule"]
        if action.tool == "read_file" and isinstance(task, SWETask):
            if isinstance(obs.result, str) and "FIXME" in obs.result:
                return action.args["file"]
        return None

    def retry(self, original: Trace, plan: RetryPlan, task) -> Trace:
        new = Trace(
            trace_id=f"retry-{original.trace_id}-{random.randint(1000,9999)}",
            benchmark=original.benchmark,
            strategy=f"{original.strategy}+retry",
            task_id=task.task_id,
        )
        for s in original.steps:
            new.add_step(s)  # carry forward (don't re-pay cost mentally)
        # Track best LR for MLE
        best_lr, best_loss = None, float("inf")
        for s in original.steps:
            if s.action.tool == "train" and isinstance(s.observation.result, (int, float)):
                if s.observation.result < best_loss:
                    best_loss, best_lr = s.observation.result, s.action.args.get("lr")

        for pa in plan.actions:
            obs = self._execute(pa.concrete_action, task)
            new.add_step(Step(len(new.steps), f"RETRY: {pa.description}",
                              pa.concrete_action, obs,
                              f"[suggested] {pa.rationale}"))
            ans = self._extract_answer(pa.concrete_action, obs, task)
            if ans is not None and task.is_correct(ans):
                new.final_answer = ans; new.succeeded = True
                break
            if pa.concrete_action.tool == "train":
                lr = pa.concrete_action.args.get("lr")
                if isinstance(obs.result, (int,float)) and obs.result < best_loss:
                    best_loss, best_lr = obs.result, lr
        else:
            if isinstance(task, MLETask) and best_lr is not None:
                new.final_answer = best_lr
                new.succeeded = task.is_correct(best_lr)
            else:
                new.final_answer = original.final_answer
                new.succeeded = original.succeeded
        score_trace(new)
        return new

    # ----- 4. Closed loop -----
    def evaluate_with_retry(self, strategy_fn, task, benchmark, max_retries=3):
        initial = strategy_fn(task); score_trace(initial)
        history = [initial]; current = initial
        plans = []
        for _ in range(max_retries):
            if current.succeeded:
                break
            plan = self.suggest(current, task); plans.append(plan)
            if not plan.actions:
                break
            current = self.retry(current, plan, task)
            history.append(current)
        best = max(history, key=lambda t: t.scores["weighted_total"])
        return {"history": history, "best": best, "plans": plans}


evaluator = TraceEvaluator()
print("TraceEvaluator ready.")


### 12.1 Demo: rescue a deliberately-failing strategy

`arc_greedy` always commits to `identity` (fails on a `rotate_90` task).
`mle_greedy` commits to `lr=1e-2` (off-optimum). The evaluator should fix both.

In [ ]:
print("=== ARC: greedy (always fails) -> retry loop ===")
arc_task = ARCTask()
arc_run = evaluator.evaluate_with_retry(arc_greedy, arc_task, Benchmark.ARC, max_retries=3)
for i, t in enumerate(arc_run["history"]):
    tag = "INITIAL" if i == 0 else f"RETRY {i}"
    print(f"  {tag:9s} strategy={t.strategy:20s} "
          f"steps={len(t.steps):2d} ok={t.succeeded} "
          f"answer={t.final_answer!r} total={t.scores['weighted_total']:.2f}")

if arc_run["plans"]:
    print(f"\n  First retry plan: {arc_run['plans'][0].summary}")
    print(f"  Diagnosed:")
    for d in arc_run["plans"][0].diagnosis:
        print(f"    [{d.category} sev={d.severity:.2f}] {d.evidence}")
    print(f"  Top-3 suggested actions:")
    for a in arc_run["plans"][0].actions[:3]:
        print(f"    gain={a.expected_gain:.2f}  {a.description}")
        print(f"        action={a.concrete_action.tool}({a.concrete_action.args})")
        print(f"        why: {a.rationale}")

print("\n=== MLE: greedy (commits to lr=1e-2) -> retry loop ===")
mle_task = MLETask()
mle_run = evaluator.evaluate_with_retry(mle_greedy, mle_task, Benchmark.MLE, max_retries=3)
for i, t in enumerate(mle_run["history"]):
    tag = "INITIAL" if i == 0 else f"RETRY {i}"
    print(f"  {tag:9s} ok={t.succeeded} answer={t.final_answer!r} "
          f"steps={len(t.steps):2d} total={t.scores['weighted_total']:.2f}")

print("\n=== SWE: alphabetical greedy -> retry loop ===")
swe_task = SWETask()
swe_run = evaluator.evaluate_with_retry(swe_greedy, swe_task, Benchmark.SWE, max_retries=3)
for i, t in enumerate(swe_run["history"]):
    tag = "INITIAL" if i == 0 else f"RETRY {i}"
    print(f"  {tag:9s} ok={t.succeeded} answer={t.final_answer!r} "
          f"steps={len(t.steps):2d} total={t.scores['weighted_total']:.2f}")


### 12.2 Retry-effectiveness sweep

Run every starting strategy with the retry loop on top, measure how often the
loop turns a failure into a success.

In [ ]:
rows = []
suites = [
    (ARCTask(), ARC_STRATEGIES, Benchmark.ARC),
    (SWETask(), SWE_STRATEGIES, Benchmark.SWE),
    (MLETask(), MLE_STRATEGIES, Benchmark.MLE),
]
for task, strategies, bm in suites:
    for name, fn in strategies.items():
        run = evaluator.evaluate_with_retry(fn, task, bm, max_retries=3)
        init = run["history"][0]; best = run["best"]
        rows.append({
            "benchmark": bm.value, "strategy": name,
            "initial_ok": init.succeeded,
            "final_ok": best.succeeded,
            "rescued": (not init.succeeded) and best.succeeded,
            "retries_used": len(run["history"]) - 1,
            "initial_score": init.scores["weighted_total"],
            "final_score": best.scores["weighted_total"],
        })
retry_df = pd.DataFrame(rows)
print("Initial successes :", retry_df["initial_ok"].sum(), "/", len(retry_df))
print("Final successes   :", retry_df["final_ok"].sum(), "/", len(retry_df))
print("Strategies rescued:", retry_df["rescued"].sum())
print()
retry_df.sort_values(["benchmark","rescued"], ascending=[True,False])


In [ ]:
# --- Visualize retry rescue: bar chart of initial vs final score per strategy ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, bm in zip(axes, ["arc","swe","mle"]):
    sub = retry_df[retry_df["benchmark"]==bm].sort_values("strategy")
    x = np.arange(len(sub)); w = 0.4
    ax.bar(x - w/2, sub["initial_score"], width=w, label="initial", color="#90A4AE")
    ax.bar(x + w/2, sub["final_score"], width=w, label="after retry", color="#43A047")
    ax.set_xticks(x); ax.set_xticklabels(sub["strategy"], rotation=60, ha="right", fontsize=8)
    ax.set_title(bm.upper()); ax.set_ylim(0, 10); ax.legend(fontsize=8)
axes[0].set_ylabel("weighted total")
plt.suptitle("Retry-loop effect: initial score vs after retry", y=1.02)
plt.tight_layout(); plt.show()


### 12.3 Wiring this to a real agent

The `TraceEvaluator` is task-shape agnostic — its public surface is
`diagnose / suggest / retry / evaluate_with_retry`. To plug into a real agent:

1. Implement `task.is_correct(answer)` (gold check, e.g. `pytest` for SWE-Bench).
2. Implement `task` methods the suggested actions reference (`grep`, `read_file`,
   `train`, etc.) — these become real shell/HF calls in production.
3. Replace the per-task suggestion blocks in `TraceEvaluator.suggest` with an LLM
   call: feed the diagnosis + last K steps to `llm_anthropic(...)` and ask it to
   emit JSON of `{tool, args, rationale, expected_gain}` objects. The notebook's
   schema (`Action`, `RetryAction`) is already designed to accept that.
4. Optional: cache `(task_id, frozenset(actions_tried))` -> retry-plan to avoid
   re-suggesting the same retry across runs.

## 13. What's Real, What's Mock, How to Extend

**Real:**
- 30 distinct executable trajectories (10 strategies x 3 benchmarks).
- Working scoring across 5 weighted dimensions.
- Beam-search prune over executable rule trials.
- Hybrid trace synthesis from Top-3.
- Tree visualization, score heatmap, JSON export.

**Mock (deliberately tiny so it runs in seconds):**
- ARC: 8 candidate transformation rules on a fixed 3x3 grid.
- SWE: 5-file repo with embedded `FIXME` ground-truth.
- MLE: 5-point grid search around a known-optimal LR.

**Extending to real benchmarks:**
1. Replace `ARCTask` with a loader for the actual ARC-AGI JSONs and DSL solver.
2. Replace `SWETask` with `SWE-bench` or `SWE-bench-verified` task instances; turn
   `read_file/grep` into real shell tools, and call `pytest` for `check`.
3. Replace `MLETask` with an MLE-Bench environment and wire `train` to a small HF
   `Trainer` run.
4. Swap `arc_*` / `swe_*` / `mle_*` strategy bodies for prompts to one of the LLM
   adapters in section 11. The Trace schema stays identical.

**Scaling beam search to LLM-driven traces:**
- Replace `step_score` with an LLM critic call (`llm_anthropic` etc.) returning a
  numeric score for the partial trajectory.
- Increase `beam_width` to 5–10 and `max_depth` to repo depth.
- Cache partial traces by `(benchmark, hash(tried_actions))` to avoid recompute.

**Public trace datasets to bootstrap from (real ones, not the mocks):**
- `princeton-nlp/SWE-bench` and `princeton-nlp/SWE-bench_Verified` on Hugging Face.
- `arcprize/ARC-AGI` (input/output grids only — traces must be reconstructed).
- SWE-agent + OpenDevin run logs in their respective GitHub repos.
- `OpenAI/MLE-Bench` (announced; full trace corpus is research-grade).

There is currently **no unified open trace dataset across all three benchmarks** —
this notebook is a scaffold to *build* one.
